In [1]:
import pandas as pd


In [6]:
df = pd.read_csv("Institutional_data_full.csv")

In [8]:
print(df.columns)


Index(['Country', 'Year', 'Control_of_Corruption', 'Government_Effectiveness',
       'Political_Stability', 'Regulatory_Quality', 'Rule_of_Law',
       'Voice_and_Accountability'],
      dtype='str')


In [9]:
import pandas as pd

# Columns to check for missingness (UPDATED)
value_cols = [
    'Control_of_Corruption',
    'Government_Effectiveness',
    'Political_Stability',
    'Regulatory_Quality',
    'Rule_of_Law',
    'Voice_and_Accountability'
]

# Create working table
country_year_missing = df[['Country', 'Year'] + value_cols].copy()

# Missing count and index
country_year_missing['Missing_Count'] = country_year_missing[value_cols].isna().sum(axis=1)
country_year_missing['Missingness_Index'] = (
    country_year_missing['Missing_Count'] / len(value_cols) * 100
).round(2)

# Missingness category (same logic works for 6 variables)
def categorize_missing(count):
    if count == 0:
        return 'Complete'
    elif count == 1:
        return 'Low Missingness(1 variables)'
    elif count in [2, 3]:
        return 'Moderate Missingness(2-3 variables)'
    elif count in [4, 5]:
        return 'High Missingness(4-5 Variables)'
    else:
        return 'Fully Missing'

country_year_missing['Missingness_Category'] = country_year_missing['Missing_Count'].apply(categorize_missing)

# Create decade-style year groups (1996–2022 dataset still works fine)
country_year_missing['Year_Group'] = (
    (country_year_missing['Year'] // 10) * 10
).astype(int).astype(str) + '-' + (
    ((country_year_missing['Year'] // 10) * 10 + 9)
).astype(int).astype(str)

# Overall missingness category summary
category_summary = (
    country_year_missing['Missingness_Category']
    .value_counts()
    .reindex(['Complete', 'Low Missingness(1 variables)', 'Moderate Missingness(2-3 variables)', 'High Missingness(4-5 Variables)', 'Fully Missing'])
    .reset_index()
)
category_summary.columns = ['Missingness_Category', 'Number_of_rows']
category_summary['Percentage (%)'] = (
    category_summary['Number_of_rows'] /
    category_summary['Number_of_rows'].sum() * 100
).round(2)

# Average missingness by year group
missing_by_year_group = (
    country_year_missing.groupby('Year_Group')['Missingness_Index']
    .mean()
    .round(2)
    .reset_index(name='Average_Missingness_Index')
)
missing_by_year_group = missing_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Missingness category distribution within each year group
category_by_year_group = (
    country_year_missing.groupby(['Year_Group', 'Missingness_Category'])
    .size()
    .reset_index(name='Count')
)
category_by_year_group['Percentage (%)'] = (
    category_by_year_group.groupby('Year_Group')['Count']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)
category_by_year_group = category_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Print outputs
print("\n=== Overall Missingness Category Summary ===")
print(category_summary.to_string(index=False))

print("\n=== Average Missingness by Year Group ===")
print(missing_by_year_group.to_string(index=False))

print("\n=== Missingness Category by Year Group ===")
print(category_by_year_group.to_string(index=False))


=== Overall Missingness Category Summary ===
Missingness_Category  Number_of_rows  Percentage (%)
            Complete          5136.0           100.0
     Low Missingness             NaN             NaN
Moderate Missingness             NaN             NaN
    High Missingness             NaN             NaN
       Fully Missing             NaN             NaN

=== Average Missingness by Year Group ===
Year_Group  Average_Missingness_Index
 1990-1999                        0.0
 2000-2009                        0.0
 2010-2019                        0.0
 2020-2029                        0.0

=== Missingness Category by Year Group ===
Year_Group Missingness_Category  Count  Percentage (%)
 1990-1999             Complete    428           100.0
 2000-2009             Complete   1926           100.0
 2010-2019             Complete   2140           100.0
 2020-2029             Complete    642           100.0
